# RAG Deep-Dive Workshop — College Notes Explainer
### Retrieval-Augmented Generation from First Principles to Production
---
**Workshop Agenda (5 Phases · ~5–7 hours)**

| Phase | Topic |
|-------|-------|
| 1 | Chunking & Embeddings | 
| 2 | Document Parsing (LlamaIndex concepts) |
| 3 | Production Pipeline | 
| 4 | Project Build — Notes Explainer App | 
| 5 | Evaluation with RAGAS | 



---
## Phase 1 · Chunking, Embeddings & Semantic Meaning 

### Why do we chunk at all?

LLMs have a **context window limit** (e.g. Mistral = 8k tokens).  
A 200-page textbook will never fit. We slice it into small, retrievable pieces  
called **chunks**, then store only the *relevant* ones at query time.

> **Key insight:** Chunk quality directly controls what the model sees.  
> Garbage in → garbage answer.

### Chunking Strategies

| Strategy | How | Best for |
|----------|-----|----------|
| Fixed-size | Split every N tokens | Quick prototypes |
| Sentence-aware | Split at `.` / `\n` boundaries | General prose |
| Semantic | Split where *meaning shifts* (cosine similarity drop) | Dense technical text |


In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# !pip install langchain langchain-community faiss-cpu sentence-transformers pypdf requests streamlit


In [ ]:
# Phase 1a — Fixed-size chunking demo ─────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter

sample_text = """
Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide 
to produce oxygen and energy in the form of glucose. The light-dependent reactions 
occur in the thylakoid membranes. The Calvin cycle, also called the light-independent 
reactions, takes place in the stroma of the chloroplast. ATP and NADPH produced in the 
light reactions power the Calvin cycle to fix carbon dioxide into organic molecules.
"""

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,          # tokens per chunk
    chunk_overlap=20,        # overlap prevents losing context at boundaries
    separators=["\n\n", "\n", ".", " "],
)
chunks = splitter.split_text(sample_text)
for i, c in enumerate(chunks):
    print(f"Chunk {i+1} [{len(c)} chars]: {c[:80]}...")


### Embedding Vectors

An **embedding** is a dense N-dimensional vector that encodes the *meaning* of text.  
Similar meanings → vectors that point in the same direction.

**Distance metrics:**
- **Cosine similarity** — angle between vectors (direction only) ← most common for RAG
- **Dot product** — direction + magnitude
- **Euclidean** — raw spatial distance (sensitive to magnitude)

**Model choices for 8 GB RAM:**

| Model | Size | Best for |
|-------|------|----------|
| `all-MiniLM-L6-v2` | ~80 MB | General notes, fast |
| `BAAI/bge-small-en-v1.5` | ~130 MB | Better quality prose |
| BM25 (no model needed) | 0 MB | Keyword/code/jargon |


In [ ]:
# Phase 1b — Generate and compare embeddings ──────────────────────────────────
from langchain_community.embeddings import HuggingFaceEmbeddings
import numpy as np

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

sentences = [
    "Photosynthesis converts sunlight to glucose.",
    "Plants use light energy to make food.",        # semantically similar
    "The French Revolution began in 1789.",         # unrelated
]
vecs = embedder.embed_documents(sentences)

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"Embedding dimension : {len(vecs[0])}")
print(f"Sim(s1, s2) similar : {cosine_sim(vecs[0], vecs[1]):.4f}")
print(f"Sim(s1, s3) unrelated: {cosine_sim(vecs[0], vecs[2]):.4f}")
# Expect: similar >> unrelated


---
## Phase 2 · Document Parsing 

### Why naive PDF extraction fails

Raw `pdfminer` / `pypdf` on complex PDFs produces:
- Tables → garbled single-line strings  
- Multi-column layouts → jumbled word order  
- No section headings attached to chunks → broken citations

### What we attach to every chunk (metadata schema)

```python
{
    "source": "lecture_03.pdf",
    "page":   4,              # enables "source: page 4" citations
    "section": "3.2 Mitosis", # nearest heading (node enrichment)
}
```
For this workshop we use **LangChain's PyPDFLoader** which extracts page-level  
text and attaches `source` + `page` metadata automatically.


In [ ]:
# Phase 2 — Load a PDF and inspect metadata ───────────────────────────────────
# Replace with your own PDF path to try this live
from langchain_community.document_loaders import PyPDFLoader

# --- Uncomment and set a real path to test ---
# loader = PyPDFLoader("my_notes.pdf")
# docs   = loader.load()
# for doc in docs[:2]:
#     print("Page:", doc.metadata.get("page"))
#     print("Text preview:", doc.page_content[:200])
#     print("---")

print("Metadata attached per chunk: source, page")
print("These power citations: 'Answer from lecture_03.pdf, page 4'")


---
## Phase 3 · Production Pipeline 

### Full ingest pipeline

```
PDF/TXT  →  parse  →  chunk  →  embed  →  upsert to FAISS
```

### At query time

```
Question  →  embed question  →  similarity search  →  top-K chunks
          →  fill prompt template  →  Ollama (local LLM)  →  Answer
```

### Key production concern: Incremental reindexing

For this app (local notes) we keep it simple — re-index on new file upload.  
In production you'd:
1. SHA-256 hash each source file  
2. Compare against a SQLite sidecar  
3. Only re-embed *changed* files  

This is what separates a prototype from a production system.


In [ ]:
# Phase 3 — Build FAISS index from text chunks ────────────────────────────────
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

notes = [
    "Mitosis is cell division producing two identical daughter cells.",
    "Meiosis produces four genetically distinct gametes.",
    "DNA replication occurs during the S phase of interphase.",
    "Photosynthesis occurs in the chloroplast's thylakoid membranes.",
    "The Calvin cycle fixes CO2 into glucose in the stroma.",
]

from langchain.schema import Document
docs = [Document(page_content=t, metadata={"source": "demo", "page": i}) for i, t in enumerate(notes)]

vectorstore = FAISS.from_documents(docs, embedder)

# Similarity search
results = vectorstore.similarity_search("How does photosynthesis work?", k=2)
for r in results:
    print(f"[page {r.metadata['page']}] {r.page_content}")


In [ ]:
# Phase 3 — Call Ollama local LLM ─────────────────────────────────────────────
# Requires: ollama serve  +  ollama pull mistral
import requests, json

def ask_ollama(prompt: str, model: str = "mistral") -> str:
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=120,
    )
    return r.json().get("response", "no response")

# Test call (Ollama must be running)
# response = ask_ollama("What is RAG in one sentence?")
# print(response)
print("Ollama call function ready. Start Ollama with: ollama serve")


---
## Phase 4 · Project Build — College Notes Explainer 

> **This is the main deliverable.** The full app lives in `app.py`.

### Architecture

```
┌─────────────────────────────────────────────────────┐
│                  Streamlit UI                       │
│  ┌──────────────┐  ┌──────────────────────────────┐ │
│  │ File Upload   │  │  Chat Interface             │ │
│  │ (PDF / TXT)   │  │  (3 answer modes)           │ │
│  └──────┬───────┘  └──────────────┬───────────────┘ │
└─────────┼─────────────────────────┼────────────────-┘
          │ ingest                  │ query
          ▼                         ▼
┌─────────────────────┐   ┌──────────────────────────┐
│  LangChain Pipeline │   │  Retrieval + Generation  │
│  parse → chunk →    │   │  FAISS similarity search │
│  embed → FAISS      │   │  → Ollama (mistral)      │
└─────────────────────┘   └──────────────────────────┘
```

### Three answer modes and their prompts

| Mode | Prompt style | Use case |
|------|-------------|----------|
| Simple | Clear, easy explanation | First pass understanding |
| Exam | Point-wise, formal | Revision / exam prep |
| ELI10 | Analogies, friendly | Concept intuition |

### Build milestones checklist

- [ ] File upload → ingestion → FAISS index built  
- [ ] Similarity search returning relevant chunks  
- [ ] Ollama responding correctly  
- [ ] All 3 modes produce different formatted answers  
- [ ] Source citations show filename + page  
- [ ] Chat history persists across turns in session  


In [ ]:
# Phase 4 — End-to-end RAG function (same logic as app.py) ───────────────────
import os, tempfile, requests
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document

PROMPTS = {
    "simple": (
        "Explain using ONLY this context, clearly and simply.\n"
        "Context: {context}\nQuestion: {question}\nAnswer:"
    ),
    "exam": (
        "Write a structured, point-wise exam answer using ONLY this context.\n"
        "Context: {context}\nQuestion: {question}\nAnswer:"
    ),
    "eli10": (
        "Explain to a 10-year-old with analogies, using ONLY this context.\n"
        "Context: {context}\nQuestion: {question}\nAnswer:"
    ),
}

def full_rag_pipeline(notes_text: str, question: str, mode: str = "simple") -> str:
    # 1. Chunk
    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
    chunks = splitter.create_documents([notes_text])

    # 2. Embed + index
    embedder = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
    )
    vs = FAISS.from_documents(chunks, embedder)

    # 3. Retrieve
    docs = vs.similarity_search(question, k=3)
    context = "\n---\n".join(d.page_content for d in docs)

    # 4. Generate
    prompt = PROMPTS[mode].format(context=context, question=question)
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": "mistral", "prompt": prompt, "stream": False},
        timeout=120,
    )
    return r.json().get("response", "no response")

# --- Test (requires Ollama) ---
# answer = full_rag_pipeline(
#     notes_text="Photosynthesis converts sunlight to glucose in chloroplasts.",
#     question="What is photosynthesis?",
#     mode="eli10",
# )
# print(answer)
print("Pipeline function defined. See app.py for the full Streamlit version.")


---
## Phase 5 · Evaluation with RAGAS 

### The 4 RAGAS metrics

| Metric | Question answered | Catches |
|--------|------------------|---------|
| **Faithfulness** | Does the answer stay within retrieved context? | Hallucinations |
| **Answer Relevancy** | Does the answer address the question? | Off-topic answers |
| **Context Precision** | Are retrieved chunks relevant? | Retrieval noise |
| **Context Recall** | Did retrieval surface everything needed? | Coverage gaps |

### Evaluation sweep

Run a grid over:
- **Chunk size:** 256 / 512 / 1024 tokens  
- **Top-K:** 3 / 5 / 10 chunks  
- **Reranker:** on vs off  

Pick the combo with the best RAGAS score for your corpus.

### Lost in the Middle 🧠

> Models attend better to context at the **start and end** of the prompt.  
> → Always place the **most relevant chunk first**.  
> This is a free accuracy improvement — no model change needed.


In [ ]:
# Phase 5 — RAGAS evaluation scaffold ────────────────────────────────────────
# pip install ragas datasets

# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset

# Prepare evaluation data
eval_data = {
    "question":  ["What is photosynthesis?"],
    "answer":    ["Photosynthesis converts sunlight to glucose in chloroplasts."],
    "contexts":  [[
        "Photosynthesis occurs in the chloroplast's thylakoid membranes.",
        "The Calvin cycle fixes CO2 into glucose in the stroma.",
    ]],
    "ground_truth": ["Photosynthesis is the process of converting light energy to chemical energy."],
}

# dataset = Dataset.from_dict(eval_data)
# result  = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
# print(result)

print("RAGAS scaffold ready.")
print("Install ragas and datasets, then uncomment to run evaluation.")


---
## Quick Reference — Full Stack

| Component | Tool | Why |
|-----------|------|-----|
| Document loading | `PyPDFLoader` / `TextLoader` | Page-level metadata |
| Chunking | `RecursiveCharacterTextSplitter` | Respects sentence boundaries |
| Embeddings | `all-MiniLM-L6-v2` | 80 MB, fast, good quality |
| Vector store | FAISS (local) | No server needed, 8 GB RAM friendly |
| LLM | Ollama → Mistral / LLaMA3 | 100% local, no API cost |
| UI | Streamlit | Fast to build, easy to demo |
| Evaluation | RAGAS | Industry-standard RAG metrics |

## Next Steps / Extensions

1. **Hybrid retrieval** — combine FAISS (dense) with BM25 (sparse) using RRF  
2. **Incremental reindex** — SHA-256 hashing + SQLite sidecar  
3. **Cross-encoder reranking** — `cross-encoder/ms-marco-MiniLM-L-6-v2`  
4. **FastAPI backend** — expose `/ingest` and `/query` as REST endpoints  
5. **Swap vector store** — replace FAISS with Qdrant for production persistence  

> **Portfolio pitch:** *"Production RAG with semantic chunking, hybrid retrieval,  
> cross-encoder reranking, and incremental reindex — 100% local, zero API cost."*
